# Audience-Adaptive Content Optimizer | Evaluator-Optimizer

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
import json
import re
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
generator_llm = ChatOpenAI(model="gpt-4o")
evaluator_llm = ChatOpenAI(model="gpt-4o")

In [4]:
class EvalState(TypedDict):
    topic: str
    audience: str
    response: NotRequired[str]
    score: NotRequired[float]
    feedback: NotRequired[str]
    iteration: NotRequired[int]
    final_output: NotRequired[str]

In [5]:
SCORE_THRESHOLD = 8.0
MAX_ITERATIONS = 3

In [6]:
def parse_json(text: str):
    """Extract and parse JSON from LLM output, handling markdown fences."""
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    return json.loads(cleaned)

def generate(state: EvalState) -> dict:
    if state.get("feedback"):
        prompt = (
            f"Improve your explanation based on this feedback.\n\n"
            f"Topic: {state['topic']}\n"
            f"Target audience: {state['audience']}\n"
            f"Previous version:\n{state['response']}\n"
            f"Score: {state.get('score', 0)}/10\n"
            f"Feedback: {state['feedback']}"
        )
    else:
        prompt = (
            f"Write a clear, engaging explanation of the following topic, "
            f"tailored specifically for the target audience.\n\n"
            f"Topic: {state['topic']}\n"
            f"Target audience: {state['audience']}\n\n"
            f"Adapt your vocabulary, examples, analogies, and depth to match "
            f"what this audience would understand and find engaging."
        )

    response = generator_llm.invoke(prompt)
    return {"response": response.content, "iteration": state.get("iteration", 0) + 1}

def evaluate(state: EvalState) -> Command[Literal["generate", "finalize"]]:
    response = evaluator_llm.invoke(
        f"You are a content quality evaluator. Rate this explanation on a scale of 1-10.\n\n"
        f"Topic: {state['topic']}\n"
        f"Target audience: {state['audience']}\n"
        f"Explanation:\n{state['response']}\n\n"
        f"Evaluate on these criteria:\n"
        f"- Audience fit: Is the vocabulary, tone, and complexity appropriate for '{state['audience']}'?\n"
        f"- Clarity: Is it easy to follow with a logical flow?\n"
        f"- Accuracy: Are the facts and concepts correct?\n"
        f"- Engagement: Would this audience find it interesting?\n\n"
        f"Return JSON with 'score' (number) and 'feedback' (string with specific improvements needed)."
    )
    parsed = parse_json(response.content)
    score = float(parsed.get("score", 0))
    feedback = str(parsed.get("feedback", "No feedback provided."))
    iteration = state.get("iteration", 0)
    if score >= SCORE_THRESHOLD or iteration >= MAX_ITERATIONS:
        return Command(goto="finalize", update={"score": score, "feedback": feedback})
    return Command(goto="generate", update={"score": score, "feedback": feedback})

def finalize(state: EvalState) -> dict:
    return {"final_output": state["response"]}

In [7]:
# Build graph
graph = StateGraph(EvalState)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
# No add_conditional_edges needed -- evaluate returns Command to route directly
graph.add_edge("finalize", END)

optimizer = graph.compile()

In [8]:
# Plot the optimizer
plot_mermaid(optimizer)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	evaluate(evaluate)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	evaluate -.-> finalize;
	evaluate -.-> generate;
	generate --> evaluate;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
result = optimizer.invoke({
    "topic": "How blockchain works",
    "audience": "Senior executives with no technical background"
})
print(f"Final score: {result['score']}/10")
print(f"Iterations: {result['iteration']}")
print(result["final_output"])

Final score: 9.0/10
Iterations: 1
**Understanding Blockchain: A Senior Executive's Guide**

Imagine running your business with a ledger that records every transaction, partnership, and agreement with complete transparency and security, visible to all relevant parties but tamper-proof once entries are made. This is the basic concept behind blockchain technology.

**The Digital Ledger Analogy:**

At its core, blockchain functions like a digital ledger—think of it as a sophisticated, incorruptible Excel spreadsheet. Here’s how it works:

1. **Distributed Network:** Imagine this spreadsheet isn't on just one computer. Instead, it's duplicated across a vast network of computers, known as nodes. Every transaction is recorded simultaneously on each node, ensuring that all parties have a synchronized version of the ledger.

2. **Blocks and Chains:** Each time a transaction is made, it is packaged into a "block" with other recent transactions. Once filled, a block is verified across the network

In [10]:
stream_invoke(optimizer, {
    "topic": "How blockchain works",
    "audience": "Senior executives with no technical background"
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'topic': 'How blockchain works',
 'audience': 'Senior executives with no technical background',
 'response': '**Understanding Blockchain: A Senior Executive\'s Guide**\n\nImagine running your business with a ledger that records every transaction, partnership, and agreement with complete transparency and security, visible to all relevant parties but tamper-proof once entries are made. This is the basic concept behind blockchain technology.\n\n**The Digital Ledger Analogy:**\n\nAt its core, blockchain functions like a digital ledger—think of it as a sophisticated, incorruptible Excel spreadsheet. Here’s how it works:\n\n1. **Distributed Network:** Imagine this spreadsheet isn\'t on just one computer. Instead, it\'s duplicated across a vast network of computers, known as nodes. Every transaction is recorded simultaneously on each node, ensuring that all parties have a synchronized version of the ledger.\n\n2. **Blocks and Chains:** Each time a transaction is made, it is packaged into a "